In [1]:
!pip install transformers==4.44.2 joblib==1.4.2 scikit-learn==1.6.0 numpy==1.26.4 pandas==2.2.3 scipy==1.13.1 seaborn==0.13.2 tqdm==4.66.5 lightgbm==4.5.0 xgboost==2.1.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 75.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 103.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 46.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.9/153.9 MB 11.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.2 MB/s eta 0:00:00:00:01
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Un

In [2]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
# Load datasets
train_df = pd.read_csv('/kaggle/input/pampa-dataset/Train.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/pampa-dataset/Test.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [4]:
tokenizer = AutoTokenizer.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
model = AutoModelForSequenceClassification.from_pretrained('seyonec/ChemBERTa-zinc-base-v1', num_labels=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Custom dataset class
class SMILESDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=325):
        self.tokenizer = tokenizer
        self.dataframe = dataframe
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        smiles = self.dataframe.iloc[idx]['SMILES']
        permeability = self.dataframe.iloc[idx]['Permeability']
        inputs = self.tokenizer(smiles, return_tensors='pt', padding="max_length", truncation=True, max_length=self.max_length)
        
        input_ids = inputs['input_ids'].squeeze(0)  # Shape: (sequence_length,)
        attention_mask = inputs['attention_mask'].squeeze(0)  # Shape: (sequence_length,)
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(permeability, dtype=torch.float)
        }


# datasets
train_dataset = SMILESDataset(train_df, tokenizer)
test_dataset = SMILESDataset(test_df, tokenizer)
batch_size = 16
# data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/179M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at seyonec/ChemBERTa-zinc-base-v1 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 20

In [6]:
# Training loop
from tqdm import tqdm
for epoch in range(num_epochs):
    print(f"Entered Epoch {epoch + 1}")
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f'Training Epoch {epoch + 1}/{num_epochs}', unit='batch'):
        optimizer.zero_grad()

        # Move all batch tensors to device
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"].unsqueeze(1)  # still shape: (batch_size, 1)

        # Forward pass
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=labels
        )
        loss = outputs.loss
        train_loss += loss.item()

        # Backprop and optimizer step
        loss.backward()
        optimizer.step()

    avg_train_loss = train_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{num_epochs} - Train Loss: {avg_train_loss:.4f}')

Entered Epoch 1


Training Epoch 1/20: 100%|██████████| 348/348 [02:37<00:00,  2.21batch/s]


Epoch 1/20 - Train Loss: 0.7733
Entered Epoch 2


Training Epoch 2/20: 100%|██████████| 348/348 [02:43<00:00,  2.13batch/s]


Epoch 2/20 - Train Loss: 0.4203
Entered Epoch 3


...

Epoch 18/20 - Train Loss: 0.1623
Entered Epoch 19


Training Epoch 19/20: 100%|██████████| 348/348 [02:43<00:00,  2.13batch/s]


Epoch 19/20 - Train Loss: 0.1639
Entered Epoch 20


Training Epoch 20/20: 100%|██████████| 348/348 [02:43<00:00,  2.13batch/s]

Epoch 20/20 - Train Loss: 0.1600


In [7]:
model_name = 'ChemBERTa_model_1_pampa'
model_save_path = f'/kaggle/working/{model_name}'
os.makedirs(model_save_path, exist_ok=True)

tokenizer.save_pretrained(model_save_path)
model.save_pretrained(model_save_path)

print(f'Model and tokenizer saved to {model_save_path}')

Model and tokenizer saved to /kaggle/working/ChemBERTa_model_1_pampa


In [8]:
from scipy.stats import pearsonr, spearmanr

model.eval()
test_loss = 0
test_true_labels = []
predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing', unit='batch'):
      
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].unsqueeze(1).to(device).float()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        test_loss += loss.item()

        test_true_labels.extend(labels.cpu().numpy())
        preds = outputs.logits.squeeze().cpu().numpy()  
        predictions.extend(preds)

# Final test loss
avg_test_loss = test_loss / len(test_loader)
print(f'Test Loss: {avg_test_loss:.4f}')

test_true_labels = np.array(test_true_labels).flatten()
predictions = np.array(predictions)
print(test_true_labels.shape)
print(predictions.shape)

# Performance metrics
mse = mean_squared_error(test_true_labels, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_true_labels, predictions)
r2 = r2_score(test_true_labels, predictions)
PCC,_ = pearsonr(test_true_labels, predictions)
SCC,_ = spearmanr(test_true_labels, predictions)
# Print performance metrics
print(f'Mean Squared Error: {mse:.4f}')
print(f'Root Mean Squared Error: {rmse:.4f}')
print(f'Mean Absolute Error: {mae:.4f}')
print(f'R^2 Score: {r2:.4f}')
print(f'Pearson Correlation Coefficient: {PCC:.4f}')
print(f'Spearman Correlation Coefficient: {SCC:.4f}')

# Print hyperparameters
print("Hyperparameters:")
print(f"Learning Rate: {5e-5}")
print(f"Batch Size: 16")
print(f"Epochs: {num_epochs}")

Testing: 100%|██████████| 87/87 [00:14<00:00,  5.97batch/s]


Test Loss: 0.2977
(1392,)
(1392,)
Mean Squared Error: 0.2977
Root Mean Squared Error: 0.5456
Mean Absolute Error: 0.3850
R^2 Score: 0.5317
Pearson Correlation Coefficient: 0.7517
Spearman Correlation Coefficient: 0.7461
Hyperparameters:
Learning Rate: 5e-05
Batch Size: 16
Epochs: 20


In [9]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model_name = 'ChemBERTa_model_1_pampa'
model_save_path = f'/kaggle/working/{model_name}'

if not os.path.exists(model_save_path):
    raise FileNotFoundError(f"The model directory {model_save_path} does not exist.")

tokenizer = AutoTokenizer.from_pretrained(model_save_path)
model = AutoModel.from_pretrained(model_save_path).to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at /kaggle/working/ChemBERTa_model_1_pampa and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# Load your datasets
train_df = pd.read_csv('/kaggle/input/pampa-dataset/Train.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/pampa-dataset/Test.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [11]:
train_encodings = tokenizer(list(train_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")
test_encodings = tokenizer(list(test_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")

In [12]:
from tqdm import tqdm 
batch_size = 16 

def generate_embeddings(encodings, batch_size):
    embeddings = []
    model.eval() 
    with torch.no_grad():
        for i in tqdm(range(0, len(encodings['input_ids']), batch_size), desc="Processing batches"):
            batch = {key: val[i:i + batch_size].to(device) for key, val in encodings.items()}  
            outputs = model(**batch)
            embeddings.append(outputs.last_hidden_state)
    return torch.cat(embeddings, dim=0)

In [13]:
train_embeddings = generate_embeddings(train_encodings, batch_size)
print(train_embeddings.shape)
train_embeddings = torch.mean(train_embeddings, dim=1)
print(train_embeddings.shape)

Processing batches: 100%|██████████| 348/348 [00:37<00:00,  9.26it/s]


torch.Size([5568, 237, 768])
torch.Size([5568, 768])


In [14]:
column_names = [f'x_fine_emb_ChemBerta{i}' for i in range(train_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=train_embeddings.cpu().numpy(), columns=column_names)
train_data = pd.concat([train_df, embeddings_df], axis=1)

In [15]:
test_embeddings = generate_embeddings(test_encodings, batch_size)
print(test_embeddings.shape)
test_embeddings = torch.mean(test_embeddings, dim=1)
print(test_embeddings.shape)

Processing batches: 100%|██████████| 87/87 [00:09<00:00,  9.29it/s]

torch.Size([1392, 237, 768])
torch.Size([1392, 768])


In [16]:
column_names = [f'x_fine_emb_ChemBerta{i}' for i in range(test_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=test_embeddings.cpu().numpy(), columns=column_names)
test_data = pd.concat([test_df, embeddings_df], axis=1)

In [17]:
train_data.to_csv("/kaggle/working/Train_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_pampa.csv",index=False)
test_data.to_csv("/kaggle/working/Test_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_pampa.csv",index=False)

In [18]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [19]:
train_data = pd.read_csv("/kaggle/working/Train_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_pampa.csv")
test_data = pd.read_csv("/kaggle/working/Test_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_pampa.csv")

In [20]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -3.9)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -3.9)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df

In [21]:
X_train = train_data.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_data['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

X_test = test_data.drop(['ID','SMILES','Permeability'],axis=1)
y_test = test_data['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 768)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 768)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027365 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 768
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.028869 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 768
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1105,0.2403,0.3324,0.8225,0.9070,0.8895,0.2700,0.3644,0.5197,0.5753,0.7635,0.7513
DecisionTreeRegressor,0.2245,0.3415,0.4738,0.6395,0.8221,0.7996,0.2869,0.3815,0.5356,0.5487,0.7511,0.7331
RandomForestRegressor,0.1134,0.2431,0.3368,0.8178,0.9044,0.8860,0.2717,0.3652,0.5213,0.5726,0.7622,0.7494
GradientBoostingRegressor,0.1079,0.2392,0.3285,0.8267,0.9093,0.8913,0.2746,0.3685,0.5240,0.5681,0.7593,0.7453
AdaBoostRegressor,0.1317,0.2780,0.3629,0.7885,0.8944,0.8715,0.2784,0.3886,0.5276,0.5622,0.7528,0.7371
XGBRegressor,0.1315,0.2626,0.3627,0.7887,0.8898,0.8699,0.2726,0.3658,0.5221,0.5712,0.7624,0.7485
ExtraTreesRegressor,0.1216,0.2496,0.3487,0.8047,0.8975,0.8787,0.2735,0.3652,0.5229,0.5699,0.7617,0.7472
LinearRegression,0.1257,0.2608,0.3545,0.7981,0.8944,0.8818,0.2930,0.3766,0.5413,0.5392,0.7453,0.7386
KNeighborsRegressor,0.1415,0.2739,0.3762,0.7727,0.8812,0.8570,0.2807,0.3721,0.5298,0.5585,0.7570,0.7488
SVR,0.1118,0.2400,0.3343,0.8205,0.9060,0.8904,0.2699,0.3662,0.5195,0.5755,0.7637,0.7535


In [22]:
result_df

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1105,0.2403,0.3324,0.8225,0.9070,0.8895,0.2700,0.3644,0.5197,0.5753,0.7635,0.7513
DecisionTreeRegressor,0.2245,0.3415,0.4738,0.6395,0.8221,0.7996,0.2869,0.3815,0.5356,0.5487,0.7511,0.7331
RandomForestRegressor,0.1134,0.2431,0.3368,0.8178,0.9044,0.8860,0.2717,0.3652,0.5213,0.5726,0.7622,0.7494
GradientBoostingRegressor,0.1079,0.2392,0.3285,0.8267,0.9093,0.8913,0.2746,0.3685,0.5240,0.5681,0.7593,0.7453
AdaBoostRegressor,0.1317,0.2780,0.3629,0.7885,0.8944,0.8715,0.2784,0.3886,0.5276,0.5622,0.7528,0.7371
XGBRegressor,0.1315,0.2626,0.3627,0.7887,0.8898,0.8699,0.2726,0.3658,0.5221,0.5712,0.7624,0.7485
ExtraTreesRegressor,0.1216,0.2496,0.3487,0.8047,0.8975,0.8787,0.2735,0.3652,0.5229,0.5699,0.7617,0.7472
LinearRegression,0.1257,0.2608,0.3545,0.7981,0.8944,0.8818,0.2930,0.3766,0.5413,0.5392,0.7453,0.7386
KNeighborsRegressor,0.1415,0.2739,0.3762,0.7727,0.8812,0.8570,0.2807,0.3721,0.5298,0.5585,0.7570,0.7488
SVR,0.1118,0.2400,0.3343,0.8205,0.9060,0.8904,0.2699,0.3662,0.5195,0.5755,0.7637,0.7535


In [23]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.876992764721869, -7.0784927455834366, -6.9...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.68000882707346, -6.4051832286751935, -5.1...","[-5.690017798624517, -6.31593900505629, -5.086...","[0.03355181451337137, 0.06912641196347255, 0.0..."
1,DecisionTreeRegressor,"[-7.0, -7.0, -7.0, -7.43, -5.48, -7.0, -6.19, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.24, -6.22, -4.54, -5.58, -4.6, -5.94, -6....","[-5.718, -6.166000000000001, -5.14199999999999...","[0.38248660107250837, 0.13821722034536812, 0.4..."
2,RandomForestRegressor,"[-7.077339684740001, -7.009998052690003, -6.99...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.6865153932200005, -6.302948417950002, -5....","[-5.66337559055, -6.318153489570002, -5.182662...","[0.044835198808630385, 0.05643226149963428, 0...."
3,GradientBoostingRegressor,"[-7.045872738823115, -7.039751812626734, -6.97...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.499735842885564, -6.391387051703062, -5.1...","[-5.565718762598647, -6.3183478742746075, -5.0...","[0.05822389344202739, 0.037562835005197234, 0...."
4,AdaBoostRegressor,"[-7.099855191644599, -7.021539862679753, -7.13...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.760264226556754, -6.614804856782596, -5.1...","[-5.752751148242303, -6.497173223460147, -5.26...","[0.10645132383175758, 0.0641784303420044, 0.10..."
5,XGBRegressor,"[-7.069022, -7.413076, -6.885472, -6.310302, -...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.770345, -6.118152, -5.1445365, -5.4397583...","[-5.7233515, -6.2750397, -5.1498404, -5.572751...","[0.11930736, 0.22219977, 0.15466934, 0.0848344..."
6,ExtraTreesRegressor,"[-7.032147265970001, -7.015774582915002, -7.00...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.586754242269999, -6.276305418280004, -5.1...","[-5.626238763978665, -6.294197712424003, -5.13...","[0.027205297603729313, 0.04227565271430931, 0...."
7,LinearRegression,"[-6.555960665661324, -7.745495181071651, -7.13...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.830947102131329, -6.374415952439703, -5.0...","[-5.671734711141005, -6.323579170801116, -5.13...","[0.09836206094089148, 0.20842034419822358, 0.1..."
8,KNeighborsRegressor,"[-6.986666666666667, -6.986666666666667, -7.0,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.703333333333333, -6.546666666666667, -5.0...","[-5.740666666666667, -6.356000000000001, -4.95...","[0.10793825395413173, 0.17535234370958494, 0.0..."
9,SVR,"[-7.008462886736829, -7.209343817234841, -6.95...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.702487134562776, -6.307241826677811, -5.1...","[-5.7151091459299375, -6.342053749505277, -5.1...","[0.032936993971703785, 0.07622010678047722, 0...."


In [24]:
result_df.to_csv('/kaggle/working/Results_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_pampa.csv')
prediction_df.to_csv('/kaggle/working/Prediction_data_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_pampa.csv')